# Stage 1: Bird Detection (鳴いている/いない 二値分類)

BirdCLEF 2026のデータで「鳥が鳴いているか」を判定するモデルを学習。
推論時にStage 2(234種分類)の前段フィルタとして使い、FPを削減する。

## データ構成
- 鳴いている: train_audio (全てラベル付き → 鳥が鳴いている前提)
- 鳴いていない: train_soundscapes のラベルなし区間

## Settings
| Setting | Value |
|---------|-------|
| Accelerator | GPU T4 x2 |
| Internet | ON |

In [ ]:
# データ準備 (Colab用)
import os
if not os.path.exists('/kaggle'):
    os.environ['KAGGLE_API_TOKEN'] = 'YOUR_TOKEN_HERE'
    !pip install -q kaggle
    if not os.path.exists('/content/birdclef-2026/train.csv'):
        !kaggle competitions download -c birdclef-2026
        !unzip -q -o birdclef-2026.zip -d /content/birdclef-2026

In [ ]:
import os, gc, time
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# CONFIG
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0  # sec: テスト音声と同じ5秒窓
    n_mels: int = 128            # 二値分類なので軽量設定
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'  # 軽量B0で十分
    num_classes: int = 1         # 二値分類 (鳴いている=1, いない=0)
    in_channels: int = 3
    dropout: float = 0.3
    epochs: int = 10
    batch_size: int = 32
    lr: float = 1e-3
    lr_min: float = 1e-6
    weight_decay: float = 1e-4
    seed: int = 42
    num_workers: int = 0
    n_folds: int = 5
    fold: int = 0
    data_root: str = '/kaggle/input/competitions/birdclef-2026'
    output_dir: str = '/kaggle/working'
    @property
    def chunk_samples(self): return int(self.sr * self.chunk_duration)

cfg = Config()

# 環境判定
IS_KAGGLE = os.path.exists('/kaggle')
GDRIVE_DIR = None
if not IS_KAGGLE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        GDRIVE_DIR = '/content/drive/MyDrive/birdclef-checkpoints'
        os.makedirs(GDRIVE_DIR, exist_ok=True)
        cfg.output_dir = GDRIVE_DIR
        print(f'Colab mode: saving to {GDRIVE_DIR}')
    except Exception as e:
        print(f'Gdrive mount failed: {e}')

for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026', '/content/birdclef-2026']:
    if os.path.exists(p): cfg.data_root = p; break

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Data root: {cfg.data_root}')

In [ ]:
# データ作成: 鳴いている/いない のラベル付きデータセット
DATA_ROOT = Path(cfg.data_root)

def _parse_time(t):
    """時刻文字列 '00:01:30' や数値を秒に変換"""
    if isinstance(t, (int, float)):
        return float(t)
    t = str(t)
    if ':' in t:
        parts = t.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return float(t)

# === 鳴いている (label=1): train_audio から取得 ===
train_df = pd.read_csv(DATA_ROOT / 'train.csv')
positive_items = []
for _, row in train_df.iterrows():
    positive_items.append({
        'path': str(DATA_ROOT / 'train_audio' / row['filename']),
        'label': 1,
        'source': 'train_audio',
    })
print(f'Positive from 2026: {len(positive_items)}')

# === 過去年データからもpositive追加 (2025, 2024, 2023, 2022) ===
for past_year in ['2025', '2024', '2023', '2022']:
    past_root = None
    for p in [f'/kaggle/input/birdclef-{past_year}',
              f'/kaggle/input/competitions/birdclef-{past_year}',
              f'/content/birdclef-{past_year}']:
        if os.path.exists(os.path.join(p, 'train.csv')):
            past_root = p
            break
    if past_root is None:
        continue
    past_df = pd.read_csv(os.path.join(past_root, 'train.csv'))
    count = 0
    for _, row in past_df.iterrows():
        audio_path = os.path.join(past_root, 'train_audio', row['filename'])
        positive_items.append({
            'path': audio_path,
            'label': 1,
            'source': f'train_audio_{past_year}',
        })
        count += 1
    print(f'Positive from {past_year}: {count}')

print(f'Total positive: {len(positive_items)}')

# === 鳴いていない (label=0): train_soundscapes のラベルなし区間 ===
sc_labels = pd.read_csv(DATA_ROOT / 'train_soundscapes_labels.csv')
sc_files = sorted((DATA_ROOT / 'train_soundscapes').glob('*.ogg'))

labeled_segments = {}
for _, row in sc_labels.iterrows():
    fname = row['filename']
    if fname not in labeled_segments:
        labeled_segments[fname] = []
    labeled_segments[fname].append((_parse_time(row['start']), _parse_time(row['end'])))

negative_items = []
for sc_path in sc_files:
    fname = sc_path.name
    try:
        duration = librosa.get_duration(path=str(sc_path), sr=cfg.sr)
    except Exception:
        continue
    labeled = labeled_segments.get(fname, [])
    for start in np.arange(0, duration, cfg.chunk_duration):
        end = start + cfg.chunk_duration
        has_bird = any(s < end and e > start for s, e in labeled)
        if not has_bird:
            negative_items.append({
                'path': str(sc_path),
                'label': 0,
                'source': 'soundscape',
                'offset': start,
                'duration': cfg.chunk_duration,
            })

print(f'Negative (no bird): {len(negative_items)}')

# バランス調整
np.random.seed(cfg.seed)
if len(negative_items) >= len(positive_items):
    neg_idx = np.random.choice(len(negative_items), size=len(positive_items), replace=False)
    negative_items = [negative_items[i] for i in neg_idx]
    print(f'Negative downsampled to: {len(negative_items)}')
else:
    n_repeat = len(positive_items) - len(negative_items)
    repeat_idx = np.random.choice(len(negative_items), size=n_repeat, replace=True)
    negative_items = negative_items + [negative_items[i] for i in repeat_idx]
    print(f'Negative upsampled to: {len(negative_items)}')

all_items = positive_items + negative_items
all_labels = [item['label'] for item in all_items]
print(f'Total: {len(all_items)} (pos={sum(all_labels)}, neg={len(all_labels)-sum(all_labels)})')

In [ ]:
# MODEL: 軽量B0 + GlobalAvgPool + Binary Head
class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale)
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)
    @torch.no_grad()
    def forward(self, waveforms):
        with torch.amp.autocast('cuda', enabled=False):
            waveforms = waveforms.float()
            mel = self.db(self.mel(waveforms))
            B = mel.shape[0]
            flat = mel.reshape(B, -1)
            mn = flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
            mx = flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
            mel = (mel - mn) / (mx - mn + 1e-7)
            mel = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        return mel

class BirdDetector(nn.Module):
    """軽量な二値分類モデル。鳥が鳴いているかどうかだけを判定。"""
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=True,
            in_chans=cfg.in_channels, num_classes=0, global_pool='avg')
        self.head = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(self.backbone.num_features, 1)  # 二値分類
        )
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features).squeeze(-1)  # (B,)

print('Model defined')

In [ ]:
# DATASET
class BirdDetectionDataset(Dataset):
    def __init__(self, items, cfg, mode='train'):
        self.items = items
        self.cfg = cfg
        self.mode = mode
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            if item['source'] == 'soundscape':
                # soundscapeは特定の区間を読み込む
                wave, _ = librosa.load(item['path'], sr=self.cfg.sr, mono=True,
                                       offset=item['offset'], duration=item['duration'])
            else:
                wave, _ = librosa.load(item['path'], sr=self.cfg.sr, mono=True)
        except Exception:
            wave = np.zeros(self.cfg.chunk_samples, dtype=np.float32)
        
        target = self.cfg.chunk_samples
        if len(wave) > target:
            if self.mode == 'train':
                start = np.random.randint(0, len(wave) - target)
            else:
                start = 0
            wave = wave[start:start + target]
        elif len(wave) < target:
            wave = np.pad(wave, (0, target - len(wave)))
        
        # ピーク正規化
        peak = np.abs(wave).max()
        if peak > 0:
            wave = wave / peak
        
        label = np.float32(item['label'])
        return torch.tensor(wave, dtype=torch.float32), torch.tensor(label)

print('Dataset defined')

In [ ]:
# TRAINING
def train_one_epoch(model, loader, criterion, optimizer, mel_tf, device, epoch, scaler):
    model.train()
    mel_tf = mel_tf.to(device)
    total_loss, n = 0, len(loader)
    t0 = time.time()
    for i, (waveforms, labels) in enumerate(loader):
        waveforms, labels = waveforms.to(device), labels.to(device)
        mel = mel_tf(waveforms)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(mel)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        if (i+1) % 100 == 0 or (i+1) == n:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch+1} [{i+1}/{n}] loss={total_loss/(i+1):.4f} elapsed={elapsed/60:.1f}m', flush=True)
    return total_loss / n

@torch.no_grad()
def validate(model, loader, mel_tf, device):
    model.eval()
    mel_tf = mel_tf.to(device)
    all_preds, all_labels = [], []
    for waveforms, labels in loader:
        waveforms = waveforms.to(device)
        mel = mel_tf(waveforms)
        with torch.amp.autocast('cuda'):
            logits = model(mel)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    auc = roc_auc_score(labels, preds)
    f1 = f1_score(labels, (preds > 0.5).astype(int))
    return auc, f1

print('Training functions defined')

In [ ]:
# RUN TRAINING
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

# Fold分割
labels_array = np.array(all_labels)
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
train_idx, val_idx = list(skf.split(range(len(all_items)), labels_array))[cfg.fold]

train_items = [all_items[i] for i in train_idx]
val_items = [all_items[i] for i in val_idx]
print(f'Fold {cfg.fold}: train={len(train_items)}, val={len(val_items)}')

train_ds = BirdDetectionDataset(train_items, cfg, 'train')
val_ds = BirdDetectionDataset(val_items, cfg, 'val')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)

model = BirdDetector(cfg).to(device)
mel_transform = MelSpectrogramTransform(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr_min)
scaler = torch.amp.GradScaler('cuda')
criterion = nn.BCEWithLogitsLoss()

output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

best_auc = 0
for epoch in range(cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer,
                                 mel_transform, device, epoch, scaler)
    val_auc, val_f1 = validate(model, val_loader, mel_transform, device)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1}/{cfg.epochs} | loss={train_loss:.4f} | '
          f'val_auc={val_auc:.4f} | val_f1={val_f1:.4f} | lr={lr:.2e}', flush=True)
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_auc': val_auc,
            'val_f1': val_f1,
        }, str(output_dir / 'bird_detector_best.pt'))
        print(f'  -> New best: auc={best_auc:.4f}, f1={val_f1:.4f}')

print(f'\nBest AUC: {best_auc:.4f}')
print('Done!')

In [ ]:
# 出力確認
output_dir = Path(cfg.output_dir)
for f in sorted(output_dir.glob('*.pt')):
    sz = f.stat().st_size / 1e6
    print(f'{f.name}: {sz:.1f} MB')